# 00 Eval And Submission

Combined Kaggle-first notebook for Milestone 1.

This notebook boots the shared repo code, runs held-out validation on `train.csv`, and then generates `submission.csv` from `test.csv` using the same predictor and routing stack.


In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    for candidate in [ROOT, *ROOT.parents]:
        if (candidate / "src").exists():
            ROOT = candidate
            break

if not (ROOT / "src").exists() and Path("/kaggle/input").exists():
    for nested in Path("/kaggle/input").rglob("src"):
        if nested.is_dir() and (nested.parent / "configs" / "experiments" / "baseline_eval.yaml").exists():
            ROOT = nested.parent
            break

if not (ROOT / "src").exists():
    raise RuntimeError(
        "Could not find repo root with src/ and configs/experiments/baseline_eval.yaml. "
        "Attach the repo dataset to Kaggle and run this notebook from the first cell."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

WORKDIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else ROOT
IS_KAGGLE = Path("/kaggle").exists()
ROOT, WORKDIR, IS_KAGGLE


In [ ]:
from src import load_config
from src.data import load_eval_examples, resolve_dataset_path
from src.eval import build_predictor, evaluate_examples
from src.eval.reporting import export_handoff_bundle
from src.eval.splits import split_examples
from src.prompts import get_prompt_variant, get_prompt_variants
from src.solvers import ConservativeRouter
from src.submission import write_submission_csv

config = load_config(ROOT / "configs/experiments/baseline_eval.yaml")
if IS_KAGGLE and config["predictor"].get("type") == "heuristic":
    config = {
        **config,
        "predictor": {
            **config["predictor"],
            "type": "transformers_kaggle",
        },
    }
    print("Running on Kaggle: overriding predictor.type to transformers_kaggle")

train_path = resolve_dataset_path(
    config["data"].get("train_path"),
    fallback_filename="train.csv",
    auto_discover=config["data"].get("auto_discover", False),
    base_dir=ROOT,
)
test_path = resolve_dataset_path(
    config["data"].get("test_path"),
    fallback_filename="test.csv",
    auto_discover=config["data"].get("auto_discover", False),
    base_dir=ROOT,
)

router = ConservativeRouter(
    confidence_threshold=config["routing"]["confidence_threshold"],
    enabled_families=tuple(config["routing"].get("enabled_families", [])) or None,
)
predictor = build_predictor(config)

print("Predictor type:", config["predictor"]["type"])
print("Train file:", train_path)
print("Test file:", test_path)


In [ ]:
all_examples = load_eval_examples(train_path)
train_examples, validation_examples = split_examples(
    all_examples,
    validation_ratio=config["data"]["validation_ratio"],
    seed=config["data"]["seed"],
)
selected_examples = validation_examples or all_examples
eval_limit = config["data"].get("eval_limit")
if eval_limit is not None:
    selected_examples = selected_examples[: eval_limit]

selected_prompt_names = set(config["prompts"])
prompt_variants = [
    variant
    for variant in get_prompt_variants()
    if variant.name in selected_prompt_names
]

eval_result = evaluate_examples(
    selected_examples,
    prompt_variants=prompt_variants,
    predictor=predictor,
    router=router if config["routing"]["enabled"] else None,
    run_name=config["run_name"],
    output_dir=WORKDIR / config["reporting"]["output_dir"],
    failure_sample_size=config["reporting"].get("failure_sample_size", 5),
)
handoff_bundle = export_handoff_bundle(
    eval_result,
    WORKDIR / config["kaggle"]["output_dir"],
    selected_variant=config["selected_prompt_variant"],
)
eval_result.metrics


In [ ]:
summary_path = eval_result.artifact_paths["summary_md"]
print(summary_path.read_text(encoding="utf-8"))
print(f"Handoff bundle: {handoff_bundle}")


In [ ]:
test_examples = load_eval_examples(test_path)
selected_variant = get_prompt_variant(config["selected_prompt_variant"])

submission_result = evaluate_examples(
    test_examples,
    prompt_variants=[selected_variant],
    predictor=predictor,
    router=router if config["routing"]["enabled"] else None,
    run_name="submission_generation",
    output_dir=WORKDIR / config["kaggle"]["output_dir"],
    failure_sample_size=config["reporting"].get("failure_sample_size", 5),
)
submission_path = write_submission_csv(
    submission_result.predictions,
    WORKDIR / config["submission"]["output_csv"],
)
submission_path


In [ ]:
note_path = WORKDIR / config["kaggle"]["output_dir"] / "kaggle_run_note.json"
note_path.parent.mkdir(parents=True, exist_ok=True)
note_payload = {
    "status": "eval_and_submission_ready",
    "predictor_type": config["predictor"]["type"],
    "selected_prompt_variant": selected_variant.name,
    "train_path": str(train_path),
    "test_path": str(test_path),
    "submission_csv": str(submission_path),
    "eval_metrics": eval_result.metrics,
    "submission_route_breakdown": submission_result.metrics["route_breakdown"],
    "next_steps": [
        "Validate runtime and memory budget on the full Kaggle hardware path.",
        "Inspect submission formatting before final submit.",
        "Record the Kaggle checkpoint notes in reports/.",
    ],
}
note_path.write_text(json.dumps(note_payload, indent=2), encoding="utf-8")
print(note_path.read_text(encoding="utf-8"))
print((WORKDIR / config["submission"]["output_csv"]).read_text(encoding="utf-8"))
